In [6]:
import pandas as pd
import re
import json

# =========================
# 1. 데이터 로드
# =========================
df = pd.read_csv("RunRepeat_final.csv")

# =========================
# 2. Feature 변환 함수 (개선 버전)
# =========================

def convert_cushion(heel, forefoot):
    try:
        avg = (float(heel) + float(forefoot)) / 2
    except:
        return "medium"

    if avg >= 38:
        return "high"
    elif avg >= 28:
        return "medium"
    else:
        return "low"


def convert_stability(row):
    try:
        arch = float(row.get('Arch support', 0))
        torsion = float(row.get('Torsional rigidity', 0))

        if arch >= 4 or torsion >= 4:
            return "stability"
        else:
            return "neutral"
    except:
        return "neutral"


def convert_usage(weight):
    try:
        w = float(weight)
    except:
        return ["daily_run"]

    if w <= 230:
        return ["race"]
    elif w <= 260:
        return ["daily_run"]
    elif w <= 300:
        return ["long_run"]
    else:
        return ["recovery"]


def convert_foot_type(row):
    try:
        arch = float(row.get('Arch support', 0))

        if arch >= 4:
            return ["flat"]
        elif arch <= 2:
            return ["high_arch"]
        else:
            return ["neutral"]
    except:
        return ["neutral"]


def convert_width(x):
    if pd.isna(x):
        return False
    return "wide" in str(x).lower()


# =========================
# 3. Feature 생성
# =========================

df['cushion_level'] = df.apply(
    lambda x: convert_cushion(x.get('Heel Lab'), x.get('Forefoot Lab')), axis=1
)

df['stability'] = df.apply(convert_stability, axis=1)

df['usage'] = df['Weight Lab'].apply(convert_usage)

df['foot_type'] = df.apply(convert_foot_type, axis=1)

df['wide_fit'] = df['Width / fit'].apply(convert_width)

# =========================
# 4. 데이터 정리
# =========================

# 브랜드 추출 (name에서)
def extract_brand(name):
    name = str(name)

    if name.startswith("New Balance"):
        return "New Balance"
    elif name.startswith("ASICS"):
        return "ASICS"
    elif name.startswith("Nike"):
        return "Nike"
    elif name.startswith("Hoka"):
        return "Hoka"
    elif name.startswith("Brooks"):
        return "Brooks"
    elif name.startswith("Adidas"):
        return "Adidas"
    elif name.startswith("On"):
        return "On"
    elif name.startswith("PUMA"):
        return "PUMA"
    else:
        return name.split()[0]

df['brand'] = df['name'].apply(extract_brand)

def clean_price(p):
    if pd.isna(p):
        return 0

    s = str(p).replace(",", "").strip()

    try:
        # 원화
        if "₩" in s:
            return int(s.replace("₩", ""))

        # 달러
        elif "$" in s:
            usd = float(s.replace("$", ""))
            return int(usd * 1300)  # 환율 가정

        # 유로
        elif "€" in s:
            eur = float(s.replace("€", ""))
            return int(eur * 1400)

        # 숫자만 있는 경우
        else:
            return int(float(s))

    except:
        return 0

df['price'] = df['Price'].apply(clean_price)

# id 생성
def make_id(name):
    name = str(name).lower()
    name = re.sub(r'[^a-z0-9 ]', '', name)
    return name.replace(" ", "_")

df['id'] = df['name'].apply(make_id)

# =========================
# 5. 최종 컬럼 선택
# =========================

final_df = df[[
    'id',
    'name',
    'brand',
    'price',
    'cushion_level',
    'stability',
    'foot_type',
    'usage',
    'wide_fit'
]].dropna(subset=['name'])

# =========================
# 6. CSV 저장
# =========================

final_df.to_csv("shoes_recommendation_ready.csv", index=False, encoding="utf-8-sig")

# =========================
# 7. JSON 변환
# =========================

data = final_df.to_dict(orient="records")

for item in data:
    item["@search.action"] = "upload"

upload_data = {
    "value": data
}

with open("shoes_upload.json", "w", encoding="utf-8") as f:
    json.dump(upload_data, f, indent=2, ensure_ascii=False)

print("완료: CSV + JSON 생성됨")

완료: CSV + JSON 생성됨


In [ ]:
import pandas as pd
import re
import json
import requests
import time
import os
from dotenv import load_dotenv

# =========================
# 1. 설정 (OpenAI)
# =========================

load_dotenv()

ENDPOINT = os.environ["OPENAI_ENDPOINT"].rstrip("/")
DEPLOYMENT = os.environ["OPENAI_DEPLOYMENT"]
KEY      = os.environ["OPENAI_API_KEY"]

OPENAI_ENDPOINT = "https://{ENDPOINT}/openai/deployments/{DEPLOYMENT}/chat/completions?api-version=2025-01-01-preview"

OPENAI_HEADERS = {
    "Authorization": "Bearer {KEY}",
    "Content-Type": "application/json"
}

# =========================
# 2. 데이터 로드
# =========================
df = pd.read_csv("RunRepeat_verdict.csv")

# =========================
# 3. Feature 함수
# =========================

def convert_cushion(heel, forefoot):
    try:
        avg = (float(heel) + float(forefoot)) / 2
    except:
        return "medium"

    if avg >= 38:
        return "high"
    elif avg >= 28:
        return "medium"
    else:
        return "low"


def convert_stability(row):
    try:
        arch = float(row.get('Arch support', 0))
        torsion = float(row.get('Torsional rigidity', 0))

        if arch >= 4 or torsion >= 4:
            return "stability"
        return "neutral"
    except:
        return "neutral"


def convert_usage(weight):
    try:
        w = float(weight)
    except:
        return ["daily_run"]

    if w <= 230:
        return ["race"]
    elif w <= 260:
        return ["daily_run"]
    elif w <= 300:
        return ["long_run"]
    else:
        return ["recovery"]


def convert_foot_type(row):
    try:
        arch = float(row.get('Arch support', 0))
        if arch >= 4:
            return ["flat"]
        elif arch <= 2:
            return ["high_arch"]
        else:
            return ["neutral"]
    except:
        return ["neutral"]


def convert_width(x):
    if pd.isna(x):
        return False
    return "wide" in str(x).lower()

# =========================
# 4. Feature 생성
# =========================

df['cushion_level'] = df.apply(
    lambda x: convert_cushion(x.get('Heel Lab'), x.get('Forefoot Lab')), axis=1
)

df['stability'] = df.apply(convert_stability, axis=1)
df['usage'] = df['Weight Lab'].apply(convert_usage)
df['foot_type'] = df.apply(convert_foot_type, axis=1)
df['wide_fit'] = df['Width / fit'].apply(convert_width)

# =========================
# 5. 브랜드 + 가격 + id
# =========================

def extract_brand(name):
    name = str(name)

    if name.startswith("New Balance"):
        return "New Balance"
    elif name.startswith("ASICS"):
        return "ASICS"
    elif name.startswith("Nike"):
        return "Nike"
    elif name.startswith("Hoka"):
        return "Hoka"
    elif name.startswith("Brooks"):
        return "Brooks"
    elif name.startswith("Adidas"):
        return "Adidas"
    elif name.startswith("On"):
        return "On"
    elif name.startswith("PUMA"):
        return "PUMA"
    else:
        return name.split()[0]

df['brand'] = df['name'].apply(extract_brand)

def clean_price(p):
    if pd.isna(p):
        return 0

    s = str(p).replace(",", "").strip()

    try:
        if "₩" in s:
            return int(s.replace("₩", ""))
        elif "$" in s:
            return int(float(s.replace("$", "")) * 1300)
        elif "€" in s:
            return int(float(s.replace("€", "")) * 1400)
        else:
            return int(float(s))
    except:
        return 0

df['price'] = df['Price'].apply(clean_price)

def make_id(name):
    name = str(name).lower()
    name = re.sub(r'[^a-z0-9 ]', '', name)
    return name.replace(" ", "_")

df['id'] = df['name'].apply(make_id)

# =========================
# 6. 🔥 리뷰 요약 (핵심)
# =========================

def summarize_review(verdict, pros, cons):

    prompt = f"""
You are a running shoe expert.

Return ONLY valid JSON. No explanation.

Format:
{{
  "description": "...",
  "review_summary": "..."
}}

verdict:
{verdict}

pros:
{pros}

cons:
{cons}
"""

    body = {
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0
    }

    res = requests.post(OPENAI_ENDPOINT, headers=OPENAI_HEADERS, json=body)
    result = res.json()

    try:
        text = result["choices"][0]["message"]["content"]

        # 🔥 JSON 부분만 추출
        import re
        match = re.search(r'\{.*\}', text, re.DOTALL)

        if match:
            json_str = match.group()
            return json.loads(json_str)

        else:
            return fallback_summary(verdict, pros)

    except Exception as e:
        print("파싱 실패:", e)
        return fallback_summary(verdict, pros)

def fallback_summary(verdict, pros):

    desc = str(verdict)[:120]

    if isinstance(pros, str):
        pros_text = pros.split(",")[0:3]
        pros_text = ", ".join(pros_text)
    else:
        pros_text = ""

    return {
        "description": desc,
        "review_summary": pros_text
    }

# =========================
# 7. 실제 실행
# =========================

descriptions = []
summaries = []

for i, row in df.iterrows():

    print(f"{i+1}/{len(df)} processing...")

    result = summarize_review(
        row.get("verdict", ""),
        row.get("pros", ""),
        row.get("cons", "")
    )

    descriptions.append(result["description"])
    summaries.append(result["review_summary"])

    time.sleep(1)  # rate limit 방지

df["description"] = descriptions
df["review_summary"] = summaries

# =========================
# 8. 최종 데이터
# =========================

final_df = df[[
    'id', 'name', 'brand', 'price',
    'cushion_level', 'stability',
    'foot_type', 'usage', 'wide_fit',
    'description', 'review_summary'
]]

# =========================
# 9. CSV 생성 (Indexer용)
# =========================

final_df.to_csv(
    "shoes_indexer_ready.csv",
    index=False,
    encoding="utf-8-sig"
)

print("완료: CSV 생성됨 (Blob 업로드용)")

1/30 processing...
2/30 processing...
3/30 processing...
4/30 processing...
5/30 processing...
6/30 processing...
7/30 processing...
8/30 processing...
9/30 processing...
10/30 processing...
11/30 processing...
12/30 processing...
13/30 processing...
14/30 processing...
15/30 processing...
16/30 processing...
17/30 processing...
18/30 processing...
19/30 processing...
20/30 processing...
21/30 processing...
22/30 processing...
23/30 processing...
24/30 processing...
25/30 processing...
26/30 processing...
27/30 processing...
28/30 processing...
29/30 processing...
30/30 processing...
완료: CSV 생성됨 (Blob 업로드용)


In [ ]:
import gradio as gr
import requests
from dotenv import load_dotenv

# =========================
# 1. Azure AI Search 설정
# =========================
load_dotenv()

API_KEY = os.environ["SEARCH_API_KEY"]
ENDPOINT = os.environ["SEARCH_ENDPOINT"].rstrip("/")
INDEX_NAME = os.environ["SEARCH_INDEX"]

# =========================
# 2. 검색 함수
# =========================

def search_shoes(price, foot_type, width, usage):

    # filter 생성
    filters = [f"price le {price}"]

    if foot_type:
        filters.append(f"foot_type/any(f: f eq '{foot_type}')")

    if width == "wide":
        filters.append("wide_fit eq true")

    if usage:
        filters.append(f"usage/any(u: u eq '{usage}')")

    filter_query = " and ".join(filters)

    url = f"{ENDPOINT}/indexes/{INDEX_NAME}/docs"

    params = {
        "api-version": "2023-11-01",
        "search": "*",
        "$filter": filter_query,
        "$top": 3
    }

    headers = {
        "api-key": API_KEY
    }

    res = requests.get(url, headers=headers, params=params)
    data = res.json()

    # 결과 정리
    results = []
    for doc in data.get("value", []):
        results.append(
            f"👟 {doc['name']}\n"
            f"- 브랜드: {doc['brand']}\n"
            f"- 가격: {doc['price']}원\n"
            f"- 쿠션: {doc['cushion_level']}\n"
            f"- 안정성: {doc['stability']}\n"
        )

    if not results:
        return "조건에 맞는 신발이 없습니다."

    return "\n\n".join(results)


# =========================
# 3. Gradio UI
# =========================

with gr.Blocks() as demo:

    gr.Markdown("## 🏃 러닝화 추천 시스템")

    price = gr.Slider(50000, 500000, step=10000, label="가격 (원)")
    
    foot_type = gr.Dropdown(
        ["neutral", "flat", "high_arch"],
        label="발형"
    )

    width = gr.Dropdown(
        ["normal", "wide"],
        label="발볼"
    )

    usage = gr.Dropdown(
        ["daily_run", "long_run", "race"],
        label="러닝 목적"
    )

    btn = gr.Button("추천 받기")

    output = gr.Textbox(label="추천 결과", lines=10)

    btn.click(
        fn=search_shoes,
        inputs=[price, foot_type, width, usage],
        outputs=output
    )

# =========================
# 4. 실행
# =========================

demo.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


In [ ]:
import gradio as gr
import requests

# =========================
# 1. Azure Search 설정
# =========================

SEARCH_ENDPOINT = "https://9ai037search.search.windows.net"
SEARCH_API_KEY = "YOUR_SEARCH_API_KEY"
INDEX_NAME = "shoes"

# =========================
# 2. Azure OpenAI 설정
# =========================

OPENAI_ENDPOINT = "https://9ai037-openai.openai.azure.com/openai/deployments/9ai037-gpt-4o-mini/chat/completions?api-version=2025-01-01-preview"

OPENAI_HEADERS = {
    "Authorization": "Bearer YOUR_OPENAI_API_KEY",
    "Content-Type": "application/json"
}
def extract_conditions(prompt):

    endpoint = "https://9ai037-openai.openai.azure.com/openai/deployments/9ai037-gpt-4o-mini/chat/completions?api-version=2025-01-01-preview"

    headers = {
        "Authorization": "Bearer YOUR_OPENAI_KEY",
        "Content-Type": "application/json"
    }

    system_prompt = """
너는 사용자의 러닝화 요구사항을 구조화하는 AI다.

아래 JSON 형식으로만 출력해라.

{
  "price": int or null,
  "foot_type": "flat | neutral | high_arch | null",
  "wide_fit": true/false/null,
  "usage": "daily_run | long_run | race | null"
}

설명 없이 JSON만 출력해라.
"""

    body = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0
    }

    res = requests.post(endpoint, headers=headers, json=body)
    result = res.json()

    text = result["choices"][0]["message"]["content"]

    import json
    return json.loads(text)
    
# =========================
# 3. Azure Search 함수
# =========================

def search_shoes(price, foot_type, width, usage):

    filters = [f"price le {price}"]

    if foot_type:
        filters.append(f"foot_type/any(f: f eq '{foot_type}')")

    if width == "wide":
        filters.append("wide_fit eq true")

    if usage:
        filters.append(f"usage/any(u: u eq '{usage}')")

    filter_query = " and ".join(filters)

    url = f"{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs"

    params = {
        "api-version": "2023-11-01",
        "search": "*",
        "$filter": filter_query,
        "$top": 3
    }

    headers = {
        "api-key": SEARCH_API_KEY
    }

    res = requests.get(url, headers=headers, params=params)
    return res.json().get("value", [])

# =========================
# 4. GPT 요청 함수
# =========================

def request_openai(user_input, histories, shoes):

    # 후보 신발 텍스트 생성
    shoe_text = ""
    for i, s in enumerate(shoes):
        shoe_text += f"""
{i+1}. {s['name']}
- Cushion: {s['cushion_level']}
- Stability: {s['stability']}
- Usage: {', '.join(s['usage'])}
- Wide fit: {s['wide_fit']}
"""

    message_list = []

    # system prompt
    message_list.append({
        "role": "system",
        "content": "너는 러닝화 추천 전문가다. 사용자 조건에 맞는 신발을 친절하게 설명해라."
    })

    # 기존 대화 유지
    if histories:
        for chat in histories:
            message_list.append(chat)

    # 사용자 질문 + 데이터 포함
    message_list.append({
        "role": "user",
        "content": f"""
사용자 요청:
{user_input}

추천 후보:
{shoe_text}

각 신발이 왜 적합한지 한국어로 설명해줘.
"""
    })

    body = {
        "messages": message_list,
        "max_tokens": 512,
        "temperature": 0.5,
        "top_p": 0.95,
    }

    response = requests.post(OPENAI_ENDPOINT, headers=OPENAI_HEADERS, json=body)
    response_json = response.json()

    if "choices" not in response_json:
        print(response_json)
        return {"role": "assistant", "content": "API 오류 발생"}

    return response_json['choices'][0]['message']

# =========================
# 5. 메인 로직
# =========================

def click_send(prompt, histories, price, foot_type, width, usage):

    if histories is None:
        histories = []

    # 1. Azure Search → 후보 추출
    shoes = search_shoes(price, foot_type, width, usage)

    if not shoes:
        answer = {"role": "assistant", "content": "조건에 맞는 신발이 없습니다."}
    else:
        # 2. GPT → 설명 생성
        answer = request_openai(prompt, histories, shoes)

    histories.append({"role": "user", "content": prompt})
    histories.append(answer)

    return histories

# =========================
# 6. Gradio UI
# =========================

with gr.Blocks() as demo:

    gr.Markdown("## 🏃 러닝화 추천 챗봇")

    chatbot = gr.Chatbot(label="러닝화 추천")

    with gr.Row():
        price = gr.Slider(50000, 500000, step=10000, label="가격")

    with gr.Row():
        foot_type = gr.Dropdown(["neutral", "flat", "high_arch"], label="발형")
        width = gr.Dropdown(["normal", "wide"], label="발볼")

    with gr.Row():
        usage = gr.Dropdown(["daily_run", "long_run", "race"], label="러닝 목적")

    with gr.Row():
        input_textbox = gr.Textbox(label="요청 입력", scale=5)
        send_button = gr.Button("추천", scale=1)

    send_button.click(
        fn=click_send,
        inputs=[input_textbox, chatbot, price, foot_type, width, usage],
        outputs=[chatbot]
    ).then(lambda: "", None, input_textbox)

demo.launch()

In [36]:
import requests
import json
import gradio as gr
from openai import AzureOpenAI

# =========================
# 1. 설정 (여기만 수정)
# =========================

# Azure OpenAI
OPENAI_API_KEY = "YOUR_OPENAI_API_KEY"
OPENAI_ENDPOINT = "https://9ai037-openai.openai.azure.com"
OPENAI_DEPLOYMENT = "9ai037-gpt-4o-mini"

# Azure Search
SEARCH_ENDPOINT = "https://9ai037search.search.windows.net"
SEARCH_INDEX = "shoes"
SEARCH_API_KEY = "YOUR_SEARCH_API_KEY"


client = AzureOpenAI(
    api_key=OPENAI_API_KEY,
    api_version="2024-02-01",
    azure_endpoint=OPENAI_ENDPOINT
)
#MAX_PRICE_LIMIT = 540000

In [ ]:
# =========================
# 2. 로직 (번역, 필터, 검색)
# =========================
def translate(query):
    try:
        res = client.chat.completions.create(
            model=OPENAI_DEPLOYMENT,
            messages=[
                {"role": "system", "content": "Translate the user's running shoe inquiry into 2-3 concise English keywords. Return ONLY keywords."},
                {"role": "user", "content": query}
            ],
            temperature=0
        )
        return res.choices[0].message.content.strip()
    except:
        return query

def build_filter(foot, usage, price_val):
    mapping = {
        "중립": "neutral",
        "과내전": "overpronation",
        "데일리런": "daily_run",
        "레이스": "race"
    }

    filters = []

    if foot:
        valid_foot = [f for f in foot if f in mapping]
        if valid_foot:
            filters.append(f"foot_type eq '{mapping[valid_foot[0]]}'")

    if usage:
        u_filters = [f"usage eq '{mapping[u]}'" for u in usage if u in mapping]
        if u_filters:
            filters.append("(" + " or ".join(u_filters) + ")")

    if price_val and price_val < 540000:
        filters.append(f"price le {price_val}")

    return " and ".join(filters) if filters else None

def search(query_en, filter_query):
    url = f"{SEARCH_ENDPOINT}/indexes/{SEARCH_INDEX}/docs/search?api-version=2024-07-01"
    headers = {"Content-Type": "application/json", "api-key": SEARCH_API_KEY}

    body = {
        "search": query_en,
        "vectorQueries": [{"kind": "text", "text": query_en, "fields": "embedding", "k": 10}],
        "filter": filter_query,
        "queryType": "semantic",
        "semanticConfiguration": "semantic-config",
        "top": 10
    }

    try:
        res = requests.post(url, headers=headers, json=body)
        res.raise_for_status()
        docs = res.json().get("value", [])
        return sorted(docs, key=lambda x: x.get("@search.rerankerScore", 0), reverse=True)[:5]
    except:
        return []

# =========================
# 3. 답변 생성
# =========================
def generate_answer(query, docs, history):
    if not docs:
        return "죄송합니다. 현재 필터 조건에 맞는 제품을 찾을 수 없습니다. 예산을 높이거나 필터를 해제해 보시겠어요?"

    context = "\n".join([
        f"- {d.get('name')} | {int(d.get('price') or 0):,}원 | {d.get('description')[:100]}"
        for d in docs
    ])

    messages = [{
        "role": "system",
        "content": "러닝화 매장 매니저로서 Context를 기반으로 제품을 추천하세요. 친절한 한국어로 답변하고, 제품의 가격과 장점을 명확히 설명하세요."
    }]

    messages += history[-6:]

    messages.append({
        "role": "user",
        "content": f"[검색결과]\n{context}\n\n질문:{query}"
    })

    res = client.chat.completions.create(
        model=OPENAI_DEPLOYMENT,
        messages=messages
    )

    return res.choices[0].message.content

# =========================
# 4. 메인 채팅
# =========================
def chat_process(message, history, foot, usage, price_val):
    if history is None:
        history = []

    if not message:
        return history, ""

    query_en = translate(message)
    filter_q = build_filter(foot, usage, price_val)
    docs = search(query_en, filter_q)

    answer = generate_answer(message, docs, history)

    # 🔥 messages 형식 (핵심)
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": answer})

    return history, ""

# =========================
# 5. UI
# =========================
with gr.Blocks() as app:
    gr.Markdown("""
# 🏃‍♂️ AI 러닝화 큐레이터
사용자의 발 타입, 용도, 예산을 기반으로 러닝화를 추천합니다.
""")

    with gr.Row():
        # ===== 왼쪽 필터 =====
        with gr.Column(scale=1):
            gr.Markdown("### 🔍 필터")

            foot_type = gr.CheckboxGroup(
                ["중립", "과내전", "모름"],
                label="발 타입"
            )

            def enforce_single_foot(foot_list):
                if not foot_list:
                    return []
                return [foot_list[-1]]

            foot_type.change(enforce_single_foot, foot_type, foot_type)

            usage_type = gr.CheckboxGroup(
                ["데일리런", "레이스"],
                label="사용 용도"
            )

            price_slider = gr.Slider(
                0, 540000,
                value=540000,
                step=10000,
                label="최대 예산 (원)"
            )

        # ===== 오른쪽 채팅 =====
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label="💬 상담")

            with gr.Row():
                msg_input = gr.Textbox(
                    placeholder="예: 무릎 안 아픈 러닝화 추천",
                    scale=8
                )
                send_btn = gr.Button("전송", variant="primary", scale=2)

    send_btn.click(chat_process, [msg_input, chatbot, foot_type, usage_type, price_slider], [chatbot, msg_input])
    msg_input.submit(chat_process, [msg_input, chatbot, foot_type, usage_type, price_slider], [chatbot, msg_input])

app.launch(share=True,theme=gr.themes.Soft())



* Running on local URL:  http://127.0.0.1:7894

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


In [54]:
def test(x):
    return x

gr.Interface(fn=test, inputs="text", outputs="text").launch(share=True)

* Running on local URL:  http://127.0.0.1:7897

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
